[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Performance and Memory &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
import shutil
import timeit
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
CSV = SCRATCH / "readings.csv"
readings.to_csv(CSV, index=False)                                    # the readings as a file, to read back

print("pandas", pd.__version__, "|", len(readings), "readings, and", f"{CSV.stat().st_size:,}", "bytes in", CSV)

pandas 2.2.3 | 35040 readings, and 1,109,641 bytes in scratch/readings.csv


**1.** Print the memory of `readings["time"]` with and without `deep=True`, and say why the two agree.

In [2]:
print(readings["time"].memory_usage(), readings["time"].memory_usage(deep=True))

280452 280452


Both are 280,452 bytes, 8 for each of 35,040 times and 132 for the index. A `datetime64` column holds its values directly, with no Python objects for `deep=True` to count.

**2.** Make `station` a category, and print how many bytes the column takes with `deep=True`, and its categories.

In [3]:
station = readings["station"].astype("category")

print(station.memory_usage(deep=True))
print(station.cat.categories.tolist())

35564
['Bergen', 'Oslo', 'Svalbard', 'Tromso']


The count includes the index's 132 bytes. The four names are stored once, and each row keeps a one-byte code.

**3.** Read only `station` and `celsius` from `CSV`, with `station` as a category, and print the table's memory with `deep=True`.

In [4]:
small = pd.read_csv(CSV, usecols=["station", "celsius"], dtype={"station": "category"})

print(small.dtypes.to_dict())
print(small.memory_usage(deep=True).sum())

{'station': CategoricalDtype(categories=['Bergen', 'Oslo', 'Svalbard', 'Tromso'], ordered=False, categories_dtype=object), 'celsius': dtype('float64')}
315712


`celsius` stays `float64` here, so the table is larger than the worked example's, which read it as `float32`.

**4.** Read `CSV` in pieces of 5,000 rows, and count the readings below zero across all the pieces.

In [5]:
below_zero = 0
for piece in pd.read_csv(CSV, chunksize=5_000):
    below_zero += (piece["celsius"] < 0).sum()

print(below_zero, "| the whole table:", (readings["celsius"] < 0).sum())

12142 | the whole table: 12142


A count adds up across pieces, so the loop keeps a running total. The whole table gives the same count, as a check.

**5.** Convert every reading to kelvin, by adding 273.15, with `apply` and with one expression, check that the two agree, and print whether `apply` took more than 10 times as long.

In [6]:
def kelvin_apply():
    return readings["celsius"].apply(lambda celsius: celsius + 273.15)


def kelvin_expression():
    return readings["celsius"] + 273.15


print(kelvin_apply().equals(kelvin_expression()))

apply_time = min(timeit.repeat(kelvin_apply, number=3, repeat=5))
expression_time = min(timeit.repeat(kelvin_expression, number=3, repeat=5))
print("apply took more than 10 times as long:", apply_time > 10 * expression_time)

True
apply took more than 10 times as long: True


`equals` treats `NaN` in the same places as equal, so Svalbard's silent day does not spoil the check. Both timings are for three runs, so they compare directly.

The notebook is finished with its file, so this cell removes the scratch folder:

In [7]:
shutil.rmtree(SCRATCH)

print("scratch still there:", SCRATCH.exists())

scratch still there: False


---

&#8592; **Back to:** [Performance and Memory](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/18-performance-and-memory.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
